# Notebook 3 - reports, benchmark signals, and review gates

This notebook expands examples `05_programmatic_report.py` and `06_benchmark.py`. It demonstrates warning-code review, JSON report inspection, and a small synthetic benchmark with Matplotlib charts.

In [ ]:
import json
import tempfile
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt

from sql2sqlx import ConversionOptions, convert_directory

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists()
)
EXAMPLES = ROOT / "examples"

import importlib.util

spec = importlib.util.spec_from_file_location(
    "notebook_benchmark_helper", EXAMPLES / "_notebook_benchmark_helper.py"
)
assert spec and spec.loader
bench = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bench)

## Treat warning codes as review gates

The report is structured data. Teams can fail a CI job on selected warning codes while allowing known, intentionally reviewed fallbacks.

In [ ]:
result = convert_directory(str(EXAMPLES / "sql"))
warning_counts = Counter(warning.code for warning in result.report.warnings)
warning_counts

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
labels = list(warning_counts) or ["none"]
values = [warning_counts[label] for label in labels] if warning_counts else [0]
ax.bar(labels, values, color="#F28E2B")
ax.set_title("Warning-code counts")
ax.set_ylabel("count")
ax.tick_params(axis="x", rotation=30)
plt.show()

## Inspect report JSON

The JSON form is stable enough for dashboards, archived artifacts, and automated review rules.

In [ ]:
report = result.report.to_dict()
print(
    json.dumps(
        {
            key: report[key]
            for key in ["files_read", "statements", "actions_by_type", "refs_rewritten", "failures"]
        },
        indent=2,
    )
)

## Run a small benchmark

The helper generates a CTAS chain similar to `06_benchmark.py`. Keep notebook defaults modest so the cell is quick, then scale the parameters when you need a larger local measurement.

In [ ]:
runs = []
for files, statements in [(2, 20), (4, 20), (4, 40)]:
    with tempfile.TemporaryDirectory(prefix="sql2sqlx_nb_") as tmp:
        root = Path(tmp)
        lines = bench.generate(root, files=files, statements=statements)
        size = sum(path.stat().st_size for path in root.rglob("*.sql"))
        started = time.time()
        converted = convert_directory(str(root), options=ConversionOptions(jobs=1))
        elapsed = time.time() - started
        assert not converted.report.failures, converted.report.failures
        runs.append(
            {
                "files": files,
                "statements": statements,
                "lines": lines,
                "mb": size / 1_000_000,
                "elapsed": elapsed,
                "lines_per_second": lines / elapsed,
            }
        )
runs

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
labels = [f"{run['files']} files\n{run['statements']} stmts" for run in runs]
ax.bar(labels, [run["lines_per_second"] for run in runs], color="#B07AA1")
ax.set_title("Notebook benchmark throughput")
ax.set_ylabel("lines / second")
plt.show()